# 05 – Anomaly Detection

Identify anomalous vessel behaviour using two complementary methods:

1. **Isolation Forest** – flags multivariate outliers in speed / course space.
2. **DBSCAN** – flags positions that lie outside established route corridors.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.data_preprocessing import load_ais_csv, preprocess
from src.feature_engineering import build_features, get_feature_matrix
from src.anomaly_detection import IsolationForestDetector, DBSCANAnomalyDetector

%matplotlib inline

## 1. Load & Prepare

In [ ]:
raw = load_ais_csv('../data/sample/ais_sample.csv')
df  = preprocess(raw)
df  = build_features(df)
print(df.shape)

## 2. Isolation Forest

In [ ]:
feature_cols = ['lat', 'lon', 'sog', 'cog', 'delta_sog', 'delta_cog', 'distance_nm']
X = get_feature_matrix(df, feature_cols=feature_cols, dropna=True)

iso = IsolationForestDetector(contamination=0.03, n_estimators=100)
iso.fit(X)

df_scored = iso.flag_anomalies(df, feature_cols=feature_cols)
n_anomalies = (df_scored['anomaly'] == -1).sum()
print(f'Flagged anomalies: {n_anomalies} ({100 * n_anomalies / len(df_scored):.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spatial plot
normal  = df_scored[df_scored['anomaly'] == 1]
anomaly = df_scored[df_scored['anomaly'] == -1]
axes[0].scatter(normal['lon'],  normal['lat'],  s=5,  alpha=0.3, label='Normal')
axes[0].scatter(anomaly['lon'], anomaly['lat'], s=30, alpha=0.9,
                marker='x', color='red', label='Anomaly')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Isolation Forest – Spatial Anomalies')
axes[0].legend()

# Score distribution
df_scored['anomaly_score'].hist(bins=40, ax=axes[1], edgecolor='black')
axes[1].set_title('Anomaly Score Distribution')
axes[1].set_xlabel('Anomaly score (lower = more anomalous)')
plt.tight_layout()
plt.show()

## 3. DBSCAN Trajectory Clustering

In [ ]:
dbscan_det = DBSCANAnomalyDetector(eps=0.4, min_samples=5)
df_clustered = dbscan_det.fit_predict(df)

n_clusters = df_clustered['cluster'].nunique() - (1 if -1 in df_clustered['cluster'].values else 0)
n_noise    = (df_clustered['cluster'] == -1).sum()
print(f'Clusters: {n_clusters}   Noise (anomaly) points: {n_noise}')

In [ ]:
plt.figure(figsize=(10, 6))
cluster_ids = sorted(df_clustered['cluster'].dropna().unique())
cmap = plt.cm.get_cmap('tab20', len(cluster_ids))

for i, cid in enumerate(cluster_ids):
    subset = df_clustered[df_clustered['cluster'] == cid]
    color = 'red' if cid == -1 else cmap(i)
    size  = 40 if cid == -1 else 5
    marker = 'x' if cid == -1 else 'o'
    label = 'Anomaly (noise)' if cid == -1 else f'Cluster {int(cid)}'
    plt.scatter(subset['lon'], subset['lat'], s=size, c=[color],
                alpha=0.7, marker=marker, label=label)

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('DBSCAN Trajectory Clustering')
plt.legend(markerscale=2, fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 4. Save Isolation Forest Detector

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
iso.save('../models/anomaly_detector.joblib')
print('Detector saved to models/anomaly_detector.joblib')